In [ ]:
!pip install -U pip

!pip install transformers==4.41.2
!pip install peft==0.11.1 #This is what gives us LoraConfig and PeftModel. Without this, you can't do Lora at all
!pip install accelerate==0.30.1 # It's a HuggingFace library for handling GPU/CPU device placement, mixed precision, and distributed training. transformers uses it under the hood when you pass the device_map, without it the model loading devices wont work.
!pip install trl==0.8.6 # Transformer Reinforcement Learning library.
!pip install bitsandbytes #This the engine behind quantization. Without this , we wont be able to convert the model to 4bits and the whole QLoRA approach falls apart.

In [ ]:
import peft
import transformers

print("peft:", peft.__version__)
print("transformers:", transformers.__version__)

In [ ]:
# import necessary libraries
import os
import torch #Everything runs on this. the model,tensors,dtypes
from datasets import load_dataset #HuggingFace dataset library
from transformers import (
    AutoModelForCausalLM, #Loads an causal LM from HF
    AutoTokenizer,#loads matching tokenizer
    BitsAndBytesConfig,#configures 4bit quantization
    HfArgumentParser,# CLI argument parsing (imported but unused here)
    TrainingArguments,# all training hyperparameters
    pipeline, # inference helper
    logging, # controls hf log verbosity
)
from peft import LoraConfig, PeftModel # LoraConfig defines how the LoRA adapters are structured and the PeftModel wraps a base model with Lora weights, also sued for merging later
from trl import SFTTrainer #SFTTrainer, which is a clean wrapper around HuggingFace's Trainer specifically designed for supervised fine-tuning. Without it you'd have to write your own training loop.The training manager. Handles the loop, gradient updates, logging, saving checkpoints.

Model: https://huggingface.co/NousResearch/Llama-2-7b-chat-hf

Llama 2 chat models prompt template
```
<s>[INST]
<<SYS>> System prompt <</SYS>>
User prompt
[/INST]
Model answer </s>
```

### Prepare dataset

In [ ]:
import pandas as pd
from datasets import Dataset

In [ ]:
# Load dataset
df = pd.read_csv("/content/sentiment_analysis_dataset.csv")

In [ ]:
# Format the dataset into the Llama 2 prompt template
#Why not just feed raw text?
#Because Llama 2 Chat was instruction-tuned using a very specific prompt format. If you fine-tune it in a different format, the model gets confused about where instructions end and where answers begin. You need to match the format it was trained on.
formatted_data = df.apply(
    lambda row: {
        "text": (
            f"""<s>[INST]
            <<SYS>> Analyze the sentiment of the following text and return either 'positive' or 'negative' <</SYS>>
            {row['text']}
            [/INST]
            {row['label']}</s>
            """
        )
    },
    axis=1,
).tolist()

dataset = Dataset.from_list(formatted_data) #Converts the list of dicts into a HuggingFace Dataset object. This is required because SFTTrainer expects this format.


In [ ]:
dataset[1]

### Model fine-tuning

In [ ]:
# QLoRA configuration
compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Quantization type (fp4 or nf4)
    bnb_4bit_compute_dtype="float16", # Compute dtype for 4-bit base models
    bnb_4bit_use_double_quant=False,
)

In [ ]:
# load base model
model_name = "NousResearch/Llama-2-7b-chat-hf"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0}
)
model.config.use_cache = False
model.config.pretraining_tp = 1

In [ ]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
# load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=16, # Scaling factor for LoRA
    lora_dropout=0.1, # Dropout rate for LoRA layers
    r=64, # Rank for LoRA adapters
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
# set training parameters
training_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=0,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine"
)

In [ ]:
# set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=None,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False,
)

In [ ]:
# train model
trainer.train()

In [ ]:
# save the fine-tuned model
new_model = "Llama-2-7b-sentiment-analysis-new"
trainer.model.save_pretrained(new_model)

In [ ]:
del model, trainer

In [ ]:
import gc
gc.collect()

### Make new fine-tuned model

In [ ]:
# load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

In [ ]:
# merge base model with LoRA weights
model = PeftModel.from_pretrained(base_model, new_model)
model = model.merge_and_unload()

In [ ]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

### Push model to Huggingface Hub

In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
# log in to HuggingFace account
!huggingface-cli login

In [ ]:
model.push_to_hub("dineshpiyasamara/Llama-2-7b-hf-sentiment-analysis-new", check_pr=True)

In [ ]:
tokenizer.push_to_hub("dineshpiyasamara/Llama-2-7b-hf-sentiment-analysis-new",check_pr=True)